# Exp 027 — A1 LambdaMART reranker + wRRF (Blind-A)

**Retrieval-side single-axis change vs 021 champion (composite 0.33).**

**Prerequisite**: `colab/Build_LGBM_Features_And_Train.ipynb` has run and produced `lgbm_ranker.zip` in Drive.

**Mechanism** per Blind-A query:
1. wRRF (BM25 + dense-metadata-qwen3 + dense-lyrics-qwen3) retrieves top-100 candidates.
2. LGBM LambdaMART booster scores each candidate using 11 task-aware features (wrrf_rank, cfbpr_score, popularity, recency, tag_count, artist-in-query, plus categoricals: conversation_goal.category/specificity + user age_group/country/gender).
3. Top-20 by LGBM score go to the submission. Greedy Qwen 1.5B + stock prompt generates the response (unchanged from 021).

**Retrieval-only change** — response side is identical to 021 (LLM ≈ 3.15 expected). nDCG@20 should lift from 0.19 by +0.03–0.08 if the learned ranker genuinely adds signal over wRRF's fusion ordering.

**Target**: nDCG@20 0.19 → 0.22–0.27, composite 0.33 → 0.35–0.37 (rank 9 → 6–7).

Wall time: ~5 min on A100 (retrieval + scoring only; no multi-sample LM).

Output → `/content/prediction.zip` + Drive backup.

In [ ]:
# 1) Verify GPU.
!nvidia-smi | head -20

In [ ]:
# 2) Clone fresh-model.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026-lora-tutorial
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026-lora-tutorial
%cd /content/recsys2026-lora-tutorial
print('\n=== CODE VERSION CHECK ===')
!git log -1 --pretty=format:'commit:  %h%ndate:    %ai%nsubject: %s'
print()

In [ ]:
# 3) Install deps (+ lightgbm for the reranker).
!pip install -q -r requirements.txt lightgbm
!python -c "import torch, transformers, bm25s, lightgbm; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'lightgbm', lightgbm.__version__)"

In [ ]:
# 4) Pull trained LGBM ranker from Drive -> models/lgbm_ranker/.
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, zipfile
SRC = '/content/drive/MyDrive/recsys2026-lgbm-ranker/lgbm_ranker.zip'
assert os.path.isfile(SRC), (
    f'lgbm_ranker.zip not found at {SRC}. Run Build_LGBM_Features_And_Train.ipynb first.'
)
os.makedirs('models/lgbm_ranker', exist_ok=True)
with zipfile.ZipFile(SRC) as zf:
    zf.extractall('models/lgbm_ranker')
!ls -lh models/lgbm_ranker/
# Print val ndcg from training metadata so we know what we're shipping.
import json
with open('models/lgbm_ranker/metadata.json') as f:
    meta = json.load(f)
print(f"\nTrained ranker metadata: val_ndcg@20 = {meta.get('best_val_ndcg20'):.4f}  "
      f"(best_iter={meta.get('best_iteration')}; trained on {meta.get('trained_on_sessions')} sessions)")

In [ ]:
# 5) Experiment parameters.
TID = '027-wrrf-lgbm-qwen15b-blindsetA'
BATCH_SIZE = 16  # wRRF top-100 -> LGBM score -> top-20 is fast; batch=16 safe.
ATTN = 'sdpa'
# import os; os.environ['HF_TOKEN'] = 'hf_...'

In [ ]:
# 6) Run Blind-A inference with the LGBM reranker auto-activated by yaml.
!cd music-crs-baselines && PYTORCH_ALLOC_CONF=expandable_segments:True \
    python run_inference_blindset.py \
    --tid {TID} \
    --eval_dataset blindset_A \
    --batch_size {BATCH_SIZE} \
    --device cuda \
    --attn_implementation {ATTN}

In [ ]:
# 7) Validate + package prediction.zip.
import json, os, shutil
SRC = f'music-crs-baselines/exp/inference/blindset_A/{TID}.json'
assert os.path.isfile(SRC)
with open(SRC) as f:
    rows = json.load(f)
print(f'rows: {len(rows)}')
assert len(rows) == 80
sample = rows[0]
required = {'session_id','user_id','turn_number','predicted_track_ids','predicted_response'}
assert not (required - set(sample.keys()))
assert len(sample['predicted_track_ids']) == 20
assert sample['predicted_response'].strip()
print(f'sample response[0]: {sample["predicted_response"][:200]!r}')

stage = '/content/_stage_prediction'
shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage, exist_ok=True)
shutil.copy(SRC, os.path.join(stage, 'prediction.json'))
!cd {stage} && rm -f /content/prediction.zip && zip -q /content/prediction.zip prediction.json
!unzip -l /content/prediction.zip
print('\nprediction.zip ready.')

In [ ]:
# 8a) Browser download.
from google.colab import files
files.download('/content/prediction.zip')

In [ ]:
# 8b) Drive backup.
import os, shutil
dst = '/content/drive/MyDrive/recsys2026-predictions'
os.makedirs(dst, exist_ok=True)
shutil.copy('/content/prediction.zip', f'{dst}/{TID}__prediction.zip')
shutil.copy(f'music-crs-baselines/exp/inference/blindset_A/{TID}.json', dst)
!ls -lh {dst}

## After scoring

Expected attribution:
- **LLM ≈ 3.15** (unchanged — retrieval-only change, response side identical to 021). If materially different, retrieval shuffling affected which top-1 track got passed to the LM prompt — that's a known coupling.
- **nDCG@20 Δ vs 0.19** is the headline number. +0.03 or more clears noise; more is better.
- **LexDiv ~0.67**, **CatDiv ~0.03** should be near-unchanged.

If nDCG@20 ≥ 0.22: genuine retrieval lift. Promote 027 as new B-champ. Next experiment can stack (e.g., feature-engineer + retrain on full 15k sessions).
If nDCG@20 in [0.17, 0.22]: within noise. Look at offline metadata for signal direction.
If nDCG@20 < 0.17: LGBM ranker regressed wRRF ordering. Investigate feature drift.